# Boresha mfano kwa Microsoft Foundry

Daftari hili linafuata [mchakato wa sasa wa kurekebisha Microsoft Foundry](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/fine-tuning?WT.mc_id=academic-105485-koreyst). Linatumia **OpenAI Python SDK (v1)** inayolenga rasilimali yako ya Foundry kwenye kiama cha `/openai/v1/`, hivyo msimbo huo huo pia unafanya kazi dhidi ya jukwaa la OpenAI kwa kubadilisha usanidi wa mteja tu.

> **Masharti ya awali**
> - Mradi wa [Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst), ukiwa na nafasi ya **Mmiliki wa Foundry** (inahitajika kwa kurekebisha na kupeleka).
> - `pip install "openai>=1.0" python-dotenv`
> - Faili `.env` yenye `AZURE_OPENAI_ENDPOINT` na `AZURE_OPENAI_API_KEY` (tazama [mwongozo wa maandalizi ya kozi](./../../../00-course-setup/02-setup-local.md?WT.mc_id=academic-105485-koreyst)).
> - Mfano wa msingi unaoungwa mkono sasa kama `gpt-4.1-mini` (angalia [orodha ya mifano inayorekebishwa](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst#fine-tuning-models)).

Kurekebisha huimarisha mfano wa msingi kwa kulifundisha tena kwa mifano ya ziada inayohusiana na kazi yako. Mbinu za uhandisi wa maelekezo kama _few-shot learning_ na _retrieval-augmented generation_ huongeza maelekezo kwa data inayohusiana, lakini zinapunguzwa na dirisha la muktadha la mfano.

Kwa kurekebisha, unafundisha tena mfano mwenyewe (ukitumia mifano mingi zaidi kuliko inayofaa ndani ya dirisha la muktadha) na kuweka toleo _mafanisi_ lisilo hitaji mifano hiyo wakati wa utabiri. Hii huongeza ubora, huruhusu dirisha la muktadha kutumika kwa uhuru, na inaweza kupunguza gharama na ucheleweshaji kwa kufupisha maelekezo. Ndani ya mfumo, Foundry hutumia **LoRA (low-rank adaptation)** kwa mafunzo bora.

Foundry inaunga mkono mbinu tatu: **kurekebisha kwa kufundishwa (SFT)** - inayotumika katika mafunzo haya - pamoja na **DPO** (uli mwafaka wa upendeleo) na **RFT** (kurekebisha kwa kuimarishwa). Mchakato una hatua nne:

1. Andaa na upakie data yako ya mafunzo **na uthibitisho**.
2. Endesha kazi ya mafunzo ili kuunda mfano uliorekebishwa.
3. Angalia kazi, hakiki vitengo, na chagua eneo la kuachia.
4. Peleka mfano uliorekebishwa na ubitumie kwa utabiri.

Katika mafunzo haya tunarekebisha `gpt-4.1-mini` kwa SFT ili kuunda chatbot inayojibu maswali kuhusu jedwali la vipindi kwa limerick.

---


### Hatua 1.1: Andaa Mkusanyiko Wako wa Data

Hebu tujenge chatbot itakayokusaidia kuelewa jedwali la mzunguko wa vitu kwa kujibu maswali kuhusu kipengele kwa limerick. Katika mafunzo haya rahisi, tutaunda tu mkusanyiko wa data kufundisha mfano kwa mfano wa mifano michache ya majibu yanayoonyesha muundo unaotarajiwa wa data. Katika matumizi halisi, utahitaji kuunda mkusanyiko wa data wenye mifano mingi zaidi. Pia unaweza kutumia mkusanyiko wa data uliopo wazi (kwa eneo lako la matumizi) ikiwa upo, na kuubadilisha muundo wake kwa matumizi ya fine-tuning.

Kwa kuwa tunazingatia `gpt-4.1-mini` na kutafuta jibu la mzunguko mmoja (chat completion) tunaweza kuunda mifano kwa kutumia [muundo huu uliopendekezwa](https://platform.openai.com/docs/guides/fine-tuning/preparing-your-dataset?WT.mc_id=academic-105485-koreyst) unaoakisi mahitaji ya kukamilisha mazungumzo ya OpenAI. Ikiwa unatarajia maudhui ya mazungumzo ya mizunguko mingi, utatumia [muundo wa mfano wa mizunguko mingi](https://platform.openai.com/docs/guides/fine-tuning/multi-turn-chat-examples?WT.mc_id=academic-105485-koreyst) unaojumuisha vigezo vya `weight` kuonesha ni ujumbe gani unapaswa kutumika (au usitumike) katika mchakato wa fine-tuning.

Tutatumia muundo rahisi wa mzunguko mmoja kwa mafunzo yetu hapa. Data iko katika [muundo wa jsonl](https://jsonlines.org/?WT.mc_id=academic-105485-koreyst) na rekodi 1 kwa kila mstari, kila mmoja ukiwakilishwa kama kitu kilichoandikwa kwa muundo wa JSON. Kipande kilicho hapa chini kinaonyesha rekodi 2 kama mfano - angalia [training-data.jsonl](../../../../../18-fine-tuning/python/openai/training-data.jsonl) kwa seti kamili ya mifano (10 mifano) tutakayotumia kwa mafunzo ya fine-tuning. **Kumbuka:** Kila rekodi _inapaswa_ kufafanuliwa katika mstari mmoja (si kugawanywa kwenye mistari mingi kama kawaida katika faili ya JSON iliyoandaliwa)

```json
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Gallium"}, {"role": "assistant", "content": "Gallium, oh gallium, so light - Melts in your hand, oh what a sight - At 86 degrees - Its liquid with ease - And in semiconductors, it's out of sight"}]}
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Hydrogen"}, {"role": "assistant", "content": "Hydrogen, the first in the line - The lightest of all, so divine - It's in water, you see - And in stars, it's the key - The universe's most common sign"}]}
```

Katika matumizi halisi utahitaji seti kubwa zaidi ya mifano kwa matokeo mazuri - upatanisho utakuwa kati ya ubora wa majibu na muda/gharama za fine-tuning. Tunatumia seti ndogo ili tufanikishe fine-tuning haraka kuonyesha mchakato. Angalia [mfano huu wa OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst) kwa mafunzo ya fine-tuning yanayochukua hatua zaidi.


---

### Hatua 1.2: Pandisha seti yako ya data

Pandisha faili zako za mafunzo na uhakiki kwa Foundry kwa kutumia API ya Faili (`purpose="fine-tune"`). Kutoa **faili la uhakiki** kunaruhusu Foundry kuripoti hasara ya uhakiki ili uweze kugundua kupita kiasi kwa mafunzo.

Kabla ya kuendesha msimbo ulio hapa chini, hakikisha umefanya:
 - Kusakinisha SDK: `pip install "openai>=1.0" python-dotenv`
 - Kuunda faili `.env` yenye `AZURE_OPENAI_ENDPOINT` na `AZURE_OPENAI_API_KEY`

Msimbo huu unaunda mteja wa OpenAI aliyeelekezwa kwenye rasilimali yako ya Foundry kwenye kiunganishi cha `/openai/v1/`, kisha unapandisha faili zote na kuchapisha vitambulisho vyao.


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Point the OpenAI SDK at your Microsoft Foundry resource's /openai/v1/ endpoint.
# (For the OpenAI platform instead, use: client = OpenAI()  with OPENAI_API_KEY set.)
client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
)

training_response = client.files.create(
    file=open("./training-data.jsonl", "rb"), purpose="fine-tune"
)
validation_response = client.files.create(
    file=open("./validation-data.jsonl", "rb"), purpose="fine-tune"
)

training_file_id = training_response.id
validation_file_id = validation_response.id

print("Training file ID:", training_file_id)
print("Validation file ID:", validation_file_id)


---

### Hatua 2.1: Unda kazi ya Fine-tuning na SDK


In [ ]:
# Create the fine-tuning job.
# trainingType "GlobalStandard" is the recommended tier (lower cost, faster queue).
# Use "Standard" if you need regional data residency, or "Developer" for cheap experiments.
job = client.fine_tuning.jobs.create(
    model="gpt-4.1-mini",              # base model to fine-tune
    training_file=training_file_id,
    validation_file=validation_file_id,
    suffix="elements-limerick",        # helps you identify the resulting model
    seed=105,                          # makes the run reproducible
    extra_body={"trainingType": "GlobalStandard"},
)

job_id = job.id
print("Fine-tuning Job ID:", job_id)
print("Status:", job.status)


---

### Hatua 2.2: Angalia hali ya kazi

Hapa kuna mambo machache unayoweza kufanya na API ya `client.fine_tuning.jobs`:
- `client.fine_tuning.jobs.list(limit=<n>)` - orodhesha kazi za fine-tuning zilizopita n
- `client.fine_tuning.jobs.retrieve(<job_id>)` - pata maelezo ya kazi maalum
- `client.fine_tuning.jobs.cancel(<job_id>)` - ghairi kazi
- `client.fine_tuning.jobs.list_events(fine_tuning_job_id=<job_id>, limit=<n>)` - orodhesha matukio kutoka kwa kazi
- `client.fine_tuning.jobs.checkpoints.list(<job_id>)` - orodhesha checkpoints zinazoweza kuanzishwa (miaka michache ya mwisho)

Wakati kazi inaanza, Foundry kwanza _huthibitisha faili ya mafunzo_ ili kuhakikisha data iko katika muundo sahihi. Mafunzo hufanyika na yanaweza kuchukua kutoka dakika hadi masaa kulingana na mfano na ukubwa wa seti ya data.


In [ ]:
# List the last 10 fine-tuning jobs
client.fine_tuning.jobs.list(limit=10)

# Retrieve the current state of your fine-tuning job
client.fine_tuning.jobs.retrieve(job_id)

# List up to 10 events from the job
client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10)


In [ ]:
# Track the job status until it is complete
response = client.fine_tuning.jobs.retrieve(job_id)

print("Job ID:", response.id)
print("Status:", response.status)
print("Trained Tokens:", response.trained_tokens)


---

### Hatua 2.3: Fuata matukio kufuatilia maendeleo


In [ ]:
# Track progress in a more granular way by checking events.
# Re-run this cell until you see "The job has successfully completed".
response = client.fine_tuning.jobs.list_events(job_id)

events = response.data
events.reverse()

for event in events:
    print(event.message)


In [ ]:
# When the job finishes, the last few epochs are available as deployable checkpoints.
# Best practice: don't blindly deploy the final epoch - review the checkpoints and pick
# the one that generalizes best (watch train_loss vs. valid_loss and token accuracy).
checkpoints = client.fine_tuning.jobs.checkpoints.list(job_id)
for cp in checkpoints.data:
    print(cp.fine_tuned_model_checkpoint, "| step:", cp.step_number)


### Hatua ya 2.4: Pitia hali, vipimo, na alama za ukaguzi katika lango la Foundry


Pia unaweza kufuatilia kazi kwenye [hati ya Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) chini ya **Build > Fine-tuning**. Chagua kazi yako kuona hali yake, matukio ya mafunzo, hyperparameters, na vipimo. Tazama vipimo hivi:

- `train_loss` na `full_valid_loss` - inapaswa kupungua kwa muda.
- `train_mean_token_accuracy` na `full_valid_mean_token_accuracy` - inapaswa kuongezeka.

Ikiwa miale ya mafunzo na uthibitisho inatofautiana, huenda una-fitingi kupita kiasi - jaribu epoksi chache au kuzidisha kiwango cha ujifunzaji kidogo. Mchoro hapa chini unaonyesha aina ya paneli za hali, ujumbe, na vipimo utakavyoona (UI halisi hutofautiana kwa mtoa huduma).

![Hali ya kazi ya fine-tuning](../../../../../translated_images/sw/fine-tuned-model-status.563271727bf7bfba.webp)


Unaweza pia kuona ujumbe wa hali na takwimu kwa kusogea chini zaidi katika dashibodi ya kuona kama inavyoonyeshwa:

| Ujumbe | Takwimu |
|:---|:---|
| ![Messages](../../../../../translated_images/sw/fine-tuned-messages-panel.4ed0c2da5ea1313b.webp) |  ![Metrics](../../../../../translated_images/sw/fine-tuned-metrics-panel.700d7e4995a65229.webp)|


---

### Hatua 3.1: Pata kitambulisho cha mfano ulioboreshwa vizuri

Wakati kazi inafanikisha, `response.fine_tuned_model` inashikilia kitambulisho cha mfano wako wa desturi (kwa mfano, `gpt-4.1-mini-2025-04-14.ft-...`). Katika Azure unapaswa kisha **kuanzisha** mfano huo kabla hujaweza kuuita.


In [ ]:
# Retrieve the id of the fine-tuned model once the job has succeeded
response = client.fine_tuning.jobs.retrieve(job_id)
fine_tuned_model_id = response.fine_tuned_model
print("Fine-tuned Model ID:", fine_tuned_model_id)


### Hatua 3.2: Tumia mfano uliobinafsishwa

Tofauti na jukwaa la OpenAI (ambapo unaweza kuita moja kwa moja kitambulisho cha mfano uliobinafsishwa), Microsoft Foundry inahitaji uanze kwa **kutumia** mfano kwanza. Njia rahisi ni kupitia [lango la Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst): nenda kwenye **Jenga > Uboreshaji**, chagua kazi yako iliyokamilika, kisha chagua **Tumia**. Toa jina la utekelezaji (kwa mfano, `elements-limerick`) - jina hilo la utekelezaji ndilo utakaolitumia kutoka kwenye msimbo.

Unaweza pia kutumia kwa njia ya programu kwa kutumia REST/ARM API ya udhibiti; angalia [mwongozo wa utekelezaji](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/fine-tuning-deploy?WT.mc_id=academic-105485-koreyst). Kumbuka kuwa mfano maalum uliotumika hulipishwa kwa saa, na utekelezaji usioamilishwa hutolewa baada ya siku 15.


In [ ]:
# Once deployed, call your fine-tuned model by its DEPLOYMENT name via the Responses API.
# (On the OpenAI platform you'd pass fine_tuned_model_id directly instead.)
deployment_name = "elements-limerick"  # the name you gave the deployment in Foundry

completion = client.responses.create(
    model=deployment_name,
    input=[
        {"role": "system", "content": "You are Elle, a factual chatbot that answers questions about elements in the periodic table with a limerick"},
        {"role": "user", "content": "Tell me about Strontium"},
    ],
    store=False,
)
print(completion.output_text)


---

### Hatua 3.3: Jaribu mfano wako ulioboreshwa katika uwanja wa Foundry

Unaweza pia kujaribu mfano uliowekwa kwenye [lango la Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) **Playground** - chagua usambazaji ulioboreshwa kutoka kwenye menyu ya mifano na jaribu agizo. Tumia **ujumbe sawa wa mfumo** ambao ulifundishwa nao; ujumbe tofauti wa mfumo unaweza kubadilisha tabia.

> **Kidokezo:** Linganisha mfano ulioboreshwa na `gpt-4.1-mini` msingi pembeni kwa pembeni. Angalia jinsi mfano ulioboreshwa unarudisha majibu katika muundo wa limerick kutoka mifano yako, wakati mfano msingi unafuata tu agizo la mfumo.

**Hii ni mfano rahisi kwa makusudi kuonyesha mchakato, si dataset halisi ya dunia.** Katika uzalishaji ungebobeza kwa data halisi (kwa mfano, orodha ya bidhaa kwa huduma kwa wateja), ambapo tofauti ya ubora inaonekana zaidi - na kufanikisha ubora huo kwa uhandisi wa agizo pekee ingelipisha tokeni nyingi zaidi kwa kila simu. Kwa mfano kamili zaidi, angalia [mwongozo wa ufundishaji wa OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst) na [mafunzo ya Foundry ya ufundishaji](https://learn.microsoft.com/azure/ai-foundry/openai/tutorials/fine-tune?WT.mc_id=academic-105485-koreyst).

---


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
